# Document Classification Generator

### Authors - Shivendra Raghav (1001123853), Amirhosein Morteza (1002187551)
### CSE 5388-002

#### This files reads converts all the text files for processed synthetic dataset into a single CSV file with classification columns and metadata.

### Import the required libraries

In [2]:
import os
import json
import random
import pandas as pd

### Read all the files into a single dataframe

In [3]:
# File path for the processed dataset
processed_non_sensitive_path = "../synthetic_dataset/processed_non_sensitive"
processed_sensitive_with_placeholders_path = "../synthetic_dataset/processed_sensitive_with_placeholders"

df = pd.DataFrame(columns=["filename", "text", "sensitive"])

# Load the processed non-sensitive dataset
for filename in os.listdir(processed_non_sensitive_path):
    if filename.endswith(".txt"):
        with open(os.path.join(processed_non_sensitive_path, filename), "r") as file:
            text = file.read()
            df = df._append({"filename": filename, "text": text, "sensitive": 0}, ignore_index=True)

# Load the processed sensitive dataset with placeholders
for filename in os.listdir(processed_sensitive_with_placeholders_path):
    if filename.endswith(".txt"):
        with open(os.path.join(processed_sensitive_with_placeholders_path, filename), "r") as file:
            text = file.read()
            df = df._append({"filename": filename, "text": text, "sensitive": 1}, ignore_index=True)

# Drop empty sentences from the sentence dataframe
df = df[df["text"] != ""]

df.head()

,filename,text,sensitive
0,dasovich-j_sent_917.txt,Message-ID: <16197052.1075843205891.JavaMail.e...,0
1,technology_prompt_storage_16_1_not_processed.txt,Scaleway Object Storage Documentation\n\nIntro...,0
2,taylor-m_sent_628.txt,Message-ID: <27392847.1075860063119.JavaMail.e...,0
3,mann-k_sent_82.txt,Message-ID: <21848911.1075845920460.JavaMail.e...,0
4,business_prompt_14_1_1.txt,## Standard Operating Procedure (SOP) for Auto...,0


### Convert the text content into sentences with classification

In [4]:
sensitive_placeholders = ["<ACCESS_KEY>", "<PRIMARY_ACCESS_KEY>", "<ACCESS_TOKEN>", "<API_KEY>", "<API_SECRET>", "<API_TOKEN>", "<BANK_ACCOUNT_NUMBER>", "<BEARER_TOKEN>", "<DATABASE_CONN>", "<INTERNATIONAL_BANK_ACCOUNT_NUMBER>", "<LICENSE_PLATE>", 
"<OAUTH_ID>", "<OAUTH_SECRET>", "<PASSWORD>", "<SSN>", "<USERNAME>", "<CREDIT_CARD_NUMBER>",  "<AZURE_SQL_DATABASE_CONN>", "<AWS_DYNAMODB_DATABASE_CONN>", "<GCP_SQL_DATABASE_CONN>", 
"<GOOGLE_FIRESTORE_DATABASE_CONN>", "<MONGODB_DATABASE_CONN>", "<GCP_POSTGRES_DATABASE_CONN>", "<POSTGRESQL_DATABASE_CONN>", "<MYSQL_DATABASE_CONN>", "<ORACLE_DATABASE_CONN>", "<ELASTISEARCH_DATABASE_CONN>", "<NEO4J_DATABASE_CONN>", "<CASSANDRA_DATABASE_CONN>", "<REDIS_DATABASE_CONN>", "<MSFT_SQL_DATABASE_CONN>", "<IBM_DB2_DATABASE_CONN>", "<COUCHBASE_DATABASE_CONN>", "<SECRET_KEY>"]

### Add synthetic secrets and NER metadata

In [5]:
# Load the secrets datasets
secrets = json.load(open("../sensitive_data_values/secrets.json"))

# Load synthetic user profiles
user_profiles_df = pd.read_csv("../sensitive_data_values/profiles.csv")

sensitive_values = {
    "<PASSWORD>": secrets["Passwords"],
    "<USERNAME>": user_profiles_df["username"].tolist(),
    "<SSN>": user_profiles_df["ssn"].tolist(),
    "<BANK_ACCOUNT_NUMBER>": user_profiles_df["bank_account_number"].tolist(),
    "<INTERNATIONAL_BANK_ACCOUNT_NUMBER>": user_profiles_df["international_bank_account_number"].tolist(),
    "<LICENSE_PLATE>": user_profiles_df["license_plate"].tolist(),
    "<CREDIT_CARD_NUMBER>": user_profiles_df["credit_card_number"].tolist(),
    "<ACCESS_TOKEN>": secrets["Access_Token"],
    "<BEARER_TOKEN>": secrets["Access_Token"],
    "<ACCESS_KEY>": secrets["Access_Token"],
    "<API_KEY>": secrets["API_Key"],
    "<API_SECRET>": secrets["API_Secret"],
    "<API_TOKEN>": secrets["API_Token"],
    "<OAUTH_ID>": secrets["OAuth_Id"],
    "<OAUTH_SECRET>": secrets["OAuth_Secret"],
    "<SECRET_KEY>": secrets["API_Secret"],
    "<MONGODB_DATABASE_CONN>": secrets["MongoDB_Conn"],
    "<AZURE_SQL_DATABASE_CONN>": secrets["Azure_SQL_Conn"],
    "<AWS_DYNAMODB_DATABASE_CONN>": secrets["AWS_DynamoDB_Conn"],
    "<GCP_SQL_DATABASE_CONN>": secrets["GCP_SQL_Conn"],
    "<GOOGLE_FIRESTORE_DATABASE_CONN>": secrets["Google_Firestore_Conn"],
    "<GCP_POSTGRES_DATABASE_CONN>": secrets["GCP_Postgres_Conn"],
    "<POSTGRESQL_DATABASE_CONN>": secrets["PostgreSQL_Conn"],
    "<MYSQL_DATABASE_CONN>": secrets["MySQL_Conn"],
    "<ORACLE_DATABASE_CONN>": secrets["Oracle_Conn"],
    "<ELASTISEARCH_DATABASE_CONN>": secrets["Elasticsearch_Conn"],
    "<NEO4J_DATABASE_CONN>": secrets["Neo4j_Conn"],
    "<CASSANDRA_DATABASE_CONN>": secrets["Cassandra_Conn"],
    "<REDIS_DATABASE_CONN>": secrets["Redis_Conn"],
    "<MSFT_SQL_DATABASE_CONN>": secrets["Microsoft_SQL_Conn"],
    "<IBM_DB2_DATABASE_CONN>": secrets["IBM_Db2_Conn"],
    "<COUCHBASE_DATABASE_CONN>": secrets["Couchbase_Conn"]
    }

database_conn = sensitive_values["<MONGODB_DATABASE_CONN>"].copy()
database_conn.extend(sensitive_values["<AZURE_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<AWS_DYNAMODB_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GCP_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GOOGLE_FIRESTORE_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GCP_POSTGRES_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<POSTGRESQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<MYSQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<ORACLE_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<ELASTISEARCH_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<NEO4J_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<CASSANDRA_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<REDIS_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<MSFT_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<IBM_DB2_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<COUCHBASE_DATABASE_CONN>"])

sensitive_values["<DATABASE_CONN>"] = database_conn

### Replace Sensitive Placeholders with Synthetic Secrets

In [6]:
# Document Classifier Dataframe with Synthetic Secrets
document_secrets_df = pd.DataFrame(columns=["filename", "text", "processed_text", "sensitive", "ner_metadata"])

# Iterate over the sentence dataframe
for index, row in df.iterrows():
    filename = row["filename"]
    text = row["text"]
    sensitive = row["sensitive"]
    ner_metadata = [] 

    # Check if sentence is sensitive
    if sensitive == 1:
        processed_text = text
        for placeholder in sensitive_placeholders:
            if placeholder in processed_text:
                value = str(random.choice(sensitive_values[placeholder]))
                processed_text = processed_text.replace(placeholder, value)
                ner_metadata.append((value, {"entities": [(processed_text.index(value), processed_text.index(value) + len(value), placeholder[1:-1])]}))
    else:
        processed_text = text

    document_secrets_df = document_secrets_df._append({"filename": filename, "text": text, "processed_text": processed_text, "sensitive": sensitive, "ner_metadata": ner_metadata}, ignore_index=True)

document_secrets_df.head()

,filename,text,processed_text,sensitive,ner_metadata
0,dasovich-j_sent_917.txt,Message-ID: <16197052.1075843205891.JavaMail.e...,Message-ID: <16197052.1075843205891.JavaMail.e...,0,[]
1,technology_prompt_storage_16_1_not_processed.txt,Scaleway Object Storage Documentation\n\nIntro...,Scaleway Object Storage Documentation\n\nIntro...,0,[]
2,taylor-m_sent_628.txt,Message-ID: <27392847.1075860063119.JavaMail.e...,Message-ID: <27392847.1075860063119.JavaMail.e...,0,[]
3,mann-k_sent_82.txt,Message-ID: <21848911.1075845920460.JavaMail.e...,Message-ID: <21848911.1075845920460.JavaMail.e...,0,[]
4,business_prompt_14_1_1.txt,## Standard Operating Procedure (SOP) for Auto...,## Standard Operating Procedure (SOP) for Auto...,0,[]


In [16]:
# Export to CSV
document_secrets_df.to_csv("../synthetic_dataset/document_classified_data.csv", index=False)